# Quickstart: computing climate indicators with earthkit-climate

This tutorial provides a short, hands-on introduction to computing climate indicators using **earthkit-climate**.
It focuses on core workflows with minimal end-to-end examples, relying on sensible defaults wherever possible.

In the **earthkit** ecosystem, data retrieved via **earthkit-data** (as `Field` or `FieldList` objects) can be passed directly to **earthkit-climate** indicators. Format conversion and unit checking are handled seamlessly behind the scenes without requiring manual conversions.

We will cover three core workflows:
1. **Single input variable**: Computing a simple threshold indicator (number of hot days).
2. **Baseline climatology & anomaly indicator**: Computing daily climatologies and anomalies using **earthkit-transforms**.
3. **Percentile-based indicator**: Computing percentile thresholds and warm days (`tx90p`).


In [1]:
import earthkit.data as ekd
import earthkit.transforms as ekt

import earthkit.climate as ekc

## 1. Simple indicator with a single input variable

We begin by loading daily maximum temperature data (`tasmax`) in Kelvin using the `earthkit-climate-sample` dataset source.


In [2]:
tasmax = ekd.from_source("earthkit-climate-sample", "synthetic-daily-temperature").to_xarray()
tasmax

/home/cuadradot/predictia_projects/git/c3s-indices/earthkit-climate/src/earthkit/climate/sample_source.py:139: UserWarning: earthkit-climate-sample datasets are made available for demonstration purposes only. Files are not guaranteed to be available long-term and may change over time. Please use official channels to obtain the contained datasets reliably for other purposes.
  warnings.warn(


<xarray.DataArray 'tasmax' (time: 12053)> Size: 96kB
array([286.172175, 282.396019, 288.025119, ..., 280.591046, 285.000216,
       292.76872 ], shape=(12053,))
Coordinates:
  * time     (time) datetime64[us] 96kB 1991-01-01 1991-01-02 ... 2023-12-31
Attributes:
    units:          K
    standard_name:  air_temperature
    cell_methods:   time: maximum

We compute the annual **number of hot days** (`tx_days_above`) where maximum temperature exceeds 30°C.


In [3]:
hot_days = ekc.indicators.tx_days_above(tasmax, thresh="30 degC", freq="YS")
hot_days

<xarray.DataArray 'tx_days_above' (time: 33)> Size: 264B
array([ 2.,  8.,  9.,  7.,  3.,  5., 10.,  8.,  9.,  2.,  5.,  6.,  5.,
        7.,  9.,  7.,  7., 12.,  7., 11.,  2.,  4.,  7.,  8., 11.,  9.,
        8.,  5.,  8.,  7., 14.,  3.,  6.])
Coordinates:
  * time     (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:   time: maximum time: sum over days
    history:        [2026-09-08 11:15:40] tx_days_above: TX_DAYS_ABOVE(tasmax...
    long_name:      The number of days with maximum temperature above 30 degc
    description:    Annual number of days where daily maximum temperature exc...

> **Note on automatic unit matching**: Notice that the input temperature dataset is in Kelvin (`K`), while the threshold parameter is specified in degrees Celsius (`"30 degC"`). **earthkit-climate** automatically handles unit checking and conversion behind the scenes.


## 2. Baseline climatology and anomaly indicators

Many climate applications evaluate conditions relative to a multi-decade reference period (e.g. 1991–2020).
In the earthkit ecosystem, baseline climatologies and daily anomalies are computed using [earthkit-transforms](https://earthkit-transforms.readthedocs.io/en/latest/concepts/climatology.html).

Here we select a 30-year baseline reference period (1991–2020) to compute a daily mean climatology, and then evaluate daily temperature anomalies.


### 30-year baseline daily climatology

Compute the daily mean climatology over the 1991–2020 reference period using **earthkit-transforms**:


In [4]:
tasmax_ref = tasmax.sel(time=slice("1991-01-01", "2020-12-31"))
daily_climatology = ekt.climatology.daily_mean(tasmax_ref)
daily_climatology

<xarray.DataArray 'tasmax' (dayofyear: 366)> Size: 3kB
array([285.73867074, 285.50551371, 285.8572956 , 286.2179636 ,
       286.56301157, 286.07676692, 287.86730535, 286.50099217,
       286.58319653, 287.53281647, 288.20597296, 287.63616904,
       288.15325646, 289.08386361, 288.94539798, 289.85179709,
       289.36769162, 289.84680995, 290.06920041, 289.92792534,
       289.98376762, 290.40473261, 291.52028052, 290.59144944,
       291.63909107, 291.90839881, 292.20289726, 292.43651006,
       292.87316336, 292.62878265, 293.13722048, 293.50224506,
       292.22446702, 292.84953866, 293.99011068, 293.62800978,
       294.48155209, 294.58629311, 294.86116205, 294.87523618,
       295.10113073, 296.19070973, 295.42443755, 295.05006284,
       295.46354092, 294.75104503, 295.72699016, 295.9093226 ,
       296.95763903, 295.99644514, 295.75498876, 296.79350314,
       297.09643574, 295.89763385, 297.61000469, 298.84752495,
       296.63252104, 297.21725436, 296.75023689, 297.73038131,
       297.03624529, 297.5539207 , 297.64715776, 298.6023899 ,
       299.12654687, 299.69450437, 298.95879236, 298.3589251 ,
       298.26262358, 298.62503872, 298.29181413, 299.23204728,
       299.21600884, 299.44314551, 297.93377626, 299.1646373 ,
       299.52305241, 299.20107159, 300.51016912, 299.55914319,
...
       270.23840316, 270.58968605, 271.06302749, 270.4830361 ,
       271.27610624, 270.77078471, 272.04992398, 271.72461046,
       271.69684729, 272.1491044 , 272.04104417, 271.20329047,
       271.90192247, 270.94513081, 270.89963206, 271.67555489,
       271.49054325, 271.55695552, 272.03979741, 272.51343656,
       272.61736491, 272.84452344, 272.92049956, 273.57418283,
       273.4155243 , 273.1340416 , 274.31431972, 273.57940753,
       274.60515806, 273.72622682, 274.1435511 , 273.53351856,
       275.16213447, 275.39792728, 275.25461092, 274.82938564,
       275.90468663, 275.94094776, 276.40916099, 275.07792546,
       276.56704115, 276.40819221, 276.79512216, 276.63450146,
       277.19575824, 276.56985897, 278.17332215, 277.00471917,
       278.12572316, 277.31444265, 278.27842774, 278.63438211,
       278.16820177, 277.87318465, 279.39307568, 279.75705951,
       278.20509789, 279.61174765, 280.24299163, 281.12698485,
       281.03095238, 281.6883533 , 281.09989884, 281.26092703,
       281.24118959, 282.14486309, 282.77525867, 282.20229142,
       282.64587539, 282.86608624, 282.99519741, 283.58906082,
       284.20014181, 283.73419526, 285.15199018, 285.16363709,
       285.22983427, 284.77384956])
Coordinates:
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
Attributes:
    units:          K
    standard_name:  air_temperature
    cell_methods:   time: maximum

### Daily temperature anomalies

Compute daily anomalies relative to the baseline climatology using **earthkit-transforms**:


In [5]:
anomalies = ekt.climatology.anomaly(tasmax, daily_climatology)
anomalies

<xarray.DataArray 'tasmax' (time: 12053)> Size: 96kB
array([ 0.43350413, -3.10949512,  2.16782349, ..., -4.5609439 ,
       -0.16342125,  7.53888552], shape=(12053,))
Coordinates:
  * time       (time) datetime64[us] 96kB 1991-01-01 1991-01-02 ... 2023-12-31
    dayofyear  (time) int64 96kB 1 2 3 4 5 6 7 8 ... 359 360 361 362 363 364 365
Attributes:
    units:          K
    standard_name:  air_temperature_anomaly
    cell_methods:   time: maximum

### Annual maximum temperature anomaly

From the daily anomalies, we compute an annual indicator metric representing the maximum daily temperature anomaly per year:


In [6]:
annual_max_anomaly = anomalies.resample(time="YS").max()
annual_max_anomaly

<xarray.DataArray 'tasmax' (time: 33)> Size: 264B
array([ 8.63658924,  7.832878  ,  9.19714499,  9.10530203,  8.45813348,
        9.19227814,  9.07940279,  9.50021493,  8.67224151,  8.98951513,
        8.98585749, 10.24386451,  8.40770552,  8.06624425, 10.20944965,
        8.53554958,  9.33877691,  8.66855342,  7.53241175,  9.43363882,
        9.94899658,  8.15232829, 10.54644895,  6.90015248,  9.51385585,
        9.47615563,  8.86787047,  8.72335733,  7.85785638,  8.10676123,
        9.93143134, 12.45688249,  8.95744653])
Coordinates:
  * time     (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
Attributes:
    units:          K
    standard_name:  air_temperature_anomaly
    cell_methods:   time: maximum

## 3. Indicator using percentiles

Percentile-based indices (such as `tx90p`, the fraction of days where maximum temperature exceeds the 90th calendar day percentile) require rolling daily percentile thresholds.

> **Note on earthkit-transforms**: The percentile calculation utility (`ekc.utils.climatology.rolling_percentiles`) is an **experimental** feature in **earthkit-climate** and will be natively integrated into **earthkit-transforms** in a future release.


### Calculate rolling 90th percentile threshold

Compute the 90th percentile threshold across a rolling 5-day window for each day of the year over the 1991–2020 reference period:


In [7]:
per90 = ekc.utils.climatology.rolling_percentiles(tasmax_ref, p=90, window_width=5)
per90

<xarray.DataArray 'tasmax' (dayofyear: 366, percentile: 1)> Size: 3kB
array([[288.85755162],
       [289.19069297],
       [289.51403079],
       [289.34283517],
       [290.15822583],
       [290.33329827],
       [290.42697651],
       [290.34287176],
       [291.00637702],
       [290.61142611],
       [291.20927317],
       [291.60191309],
       [291.90117169],
       [292.70842478],
       [292.98289481],
       [293.27140391],
       [293.41044095],
       [293.48007641],
       [293.78357645],
       [294.24361517],
...
       [283.59059711],
       [284.13682894],
       [284.87585534],
       [285.22364989],
       [285.52944054],
       [285.30184449],
       [285.47831077],
       [285.54267398],
       [285.46683785],
       [285.71342083],
       [286.20709006],
       [286.39843593],
       [286.49550279],
       [287.27590221],
       [287.71158644],
       [288.03900246],
       [288.23209355],
       [288.28108614],
       [288.63946904],
       [289.13474028]])
Coordinates:
  * dayofyear   (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * percentile  (percentile) int64 8B 90
Attributes:
    units:               K
    standard_name:       air_temperature
    cell_methods:        time: maximum
    climatology_bounds:  ['1991-01-01', '2020-12-31']
    window:              5
    alpha:               0.3333333333333333
    beta:                0.3333333333333333
    history:             [2026-09-08 11:15:40] per: percentile_doy(arr=tasmax...

### Compute annual warm days index (tx90p)

Evaluate the `tx90p` indicator using the rolling 90th percentile baseline:


In [8]:
tx90p_index = ekc.indicators.tx90p(tasmax, tasmax_per=per90, freq="YS")
tx90p_index

<xarray.DataArray 'tx90p' (time: 33, percentile: 1)> Size: 264B
array([[35.],
       [39.],
       [35.],
       [46.],
       [21.],
       [28.],
       [38.],
       [38.],
       [38.],
       [40.],
       [38.],
       [35.],
       [34.],
       [30.],
       [41.],
       [32.],
       [47.],
       [46.],
       [41.],
       [40.],
       [32.],
       [33.],
       [36.],
       [39.],
       [47.],
       [31.],
       [33.],
       [35.],
       [31.],
       [37.],
       [48.],
       [29.],
       [35.]])
Coordinates:
  * time        (time) datetime64[us] 264B 1991-01-01 1992-01-01 ... 2023-01-01
  * percentile  (percentile) int64 8B 90
Attributes:
    units:               days
    standard_name:       days_with_air_temperature_above_threshold
    cell_methods:        tasmax: time: maximum tasmax_per: time: maximum time...
    climatology_bounds:  ['1991-01-01', '2020-12-31']
    window:              5
    alpha:               0.3333333333333333
    beta:                0.3333333333333333
    history:             [2026-09-08 11:15:41] tx90p: TX90P(tasmax=tasmax, ta...
    long_name:           Number of days with maximum temperature above the 90...
    description:         Annual number of days with maximum temperature above...